# 03 - Técnicas de balanceo para series temporales CGM

Este notebook recoge una propuesta práctica para estudiar desbalanceo en eventos glucémicos raros sobre datos de Continuous Glucose Monitoring (CGM).

El contenido está organizado en tres partes:

1. Técnicas de balanceo específicas para series temporales y su justificación clínica.
2. Diseño experimental con desbalanceo artificial, pensado para simular sesgos realistas.
3. Implementaciones útiles para balanceo, generación de sesgo y evaluación con métricas de rendimiento y fairness.

La idea es que el notebook sirva como base de trabajo para los tres datasets del estudio: T1DiabetesGranada, REPLACE-BG y DiaTrend.

## 1. Técnicas de balanceo para series temporales

En problemas de CGM el desbalanceo no solo aparece entre clases. También suele existir desbalance temporal, desbalance entre pacientes y desbalance entre subgrupos clínicos. Por eso, una solución práctica suele combinar varias técnicas simples en lugar de depender de un único método complejo.

### Recomendación general

La estrategia más robusta y fácil de implementar suele ser:

- reponderación de la pérdida o focal loss para priorizar clases raras,
- muestreo balanceado por ventanas temporales para preservar contexto,
- submuestreo controlado de la clase mayoritaria para reducir redundancia,
- augmentations suaves solo en ventanas minoritarias,
- validación separada por paciente y por subgrupo para comprobar fairness.

### Técnicas propuestas

| Técnica | Qué corrige | Ventaja principal | Limitación principal |
|---|---|---|---|
| Pérdida ponderada / focal loss | Desbalanceo de clases | Muy fácil de implementar | No corrige sesgo temporal ni por subgrupo |
| Balanced mini-batches por ventanas | Clases y contexto temporal | Conserva la estructura secuencial | Puede repetir pacientes dominantes si no se controla |
| Submuestreo estratificado | Redundancia de la clase mayoritaria | Reduce coste y sesgo hacia la clase frecuente | Puede perder información útil |
| Augmentations suaves | Escasez de patrones minoritarios | Expande el conjunto sin inventar secuencias irreales | Si se exagera, genera trayectorias poco fisiológicas |
| Balanceo por paciente / grupo | Subgrupos y fairness | Evita que unos pocos pacientes dominen el entrenamiento | Puede reducir algo el rendimiento global |
| Balanceo por franja horaria | Desbalance temporal | Captura sesgos circadianos realistas | Requiere definir bien los estratos |

### Referencias útiles

- Lin et al., *Focal Loss for Dense Object Detection*.
- He y Garcia, *Learning from Imbalanced Data*.
- King y Zeng, *Logistic Regression in Rare Events Data*.
- Buda et al., *A Systematic Study of the Class Imbalance Problem in Convolutional Neural Networks*.
- Wen et al., *Time Series Data Augmentation for Deep Learning: A Survey*.
- Iwana y Uchida, *An Empirical Survey of Data Augmentation for Time Series Classification with Neural Networks*.
- Hardt et al., *Equality of Opportunity in Supervised Learning*.
- Mehrabi et al., *A Survey on Bias and Fairness in Machine Learning*.

In [6]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
 )

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CLASS_ORDER = ["hypoglycemia", "normoglycemia", "hyperglycemia"]

def find_first_existing_file(root: Path, patterns: list[str]) -> Path | None:
    for pattern in patterns:
        matches = sorted(root.glob(pattern))
        if matches:
            return matches[0]
    return None

def normalize_cgm_frame(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {
        "patient_id": "Patient_ID",
        "Patient_Id": "Patient_ID",
        "timestamp": "timestamp",
        "datetime": "timestamp",
        "measurement": "Measurement",
        "glucose": "Measurement",
        "class": "class_label",
    }
    rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=rename_map).copy()
    if "Patient_ID" in df.columns:
        df["Patient_ID"] = df["Patient_ID"].astype(str)
    if "Measurement" in df.columns:
        df["Measurement"] = pd.to_numeric(df["Measurement"], errors="coerce")

    if "timestamp" not in df.columns or df["timestamp"].isna().all():
        if "15min" in df.columns and not df["15min"].isna().all():
            df["timestamp"] = pd.to_datetime(df["15min"], errors="coerce")
        elif {"Measurement_date", "Measurement_time"}.issubset(df.columns):
            combined = df["Measurement_date"].astype(str).str.strip() + " " + df["Measurement_time"].astype(str).str.strip()
            df["timestamp"] = pd.to_datetime(combined, errors="coerce")
        elif "5min" in df.columns and not df["5min"].isna().all():
            df["timestamp"] = pd.to_datetime(df["5min"], errors="coerce")
        elif "Measurement_date" in df.columns:
            df["timestamp"] = pd.to_datetime(df["Measurement_date"], errors="coerce")
        else:
            df["timestamp"] = pd.NaT
    else:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    return df.sort_values([c for c in ["Patient_ID", "timestamp"] if c in df.columns]).reset_index(drop=True)

def assign_glucose_class(value: float) -> str:
    if pd.isna(value):
        return "missing"
    if value < 70:
        return "hypoglycemia"
    if value <= 180:
        return "normoglycemia"
    return "hyperglycemia"

def add_class_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "class_label" not in out.columns and "Measurement" in out.columns:
        out["class_label"] = out["Measurement"].apply(assign_glucose_class)
    return out

ROOT = Path.cwd()
for candidate in [ROOT] + list(ROOT.parents):
    if (candidate / "data").exists():
        ROOT = candidate
        break

DATA_DIR = ROOT / "data"
print(f"Proyecto detectado en: {ROOT}")
print(f"Carpeta data: {DATA_DIR}")

def load_cgm_dataset(dataset_name: str = "T1DiabetesGranada") -> pd.DataFrame:
    dataset_dir = DATA_DIR / dataset_name
    candidates = [
        "**/*glucose*.parquet",
        "**/*glucose*.csv",
        "**/*Glucose*.parquet",
        "**/*Glucose*.csv",
        "**/*measurements*.parquet",
        "**/*measurements*.csv",
    ]
    file_path = find_first_existing_file(dataset_dir, candidates)
    if file_path is None:
        raise FileNotFoundError(f"No se encontró un archivo de glucosa en {dataset_dir}")
    if file_path.suffix.lower() == ".parquet":
        df = pd.read_parquet(file_path)
    else:
        df = pd.read_csv(file_path)
    return add_class_labels(normalize_cgm_frame(df))

def simple_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        rows.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "nulls": int(df[col].isna().sum()),
            "n_unique": int(df[col].nunique(dropna=True))
        })
    return pd.DataFrame(rows)

Proyecto detectado en: c:\Users\Juan\Desktop\TFM_Coding
Carpeta data: c:\Users\Juan\Desktop\TFM_Coding\data


## 1.1 Implementaciones prácticas recomendadas

Las siguientes funciones están pensadas para usarse sobre un `DataFrame` con columnas equivalentes a `Patient_ID`, `timestamp`, `Measurement` y `class_label`. Si tu esquema real cambia, la función de normalización de arriba permite adaptar nombres de columnas sin tocar el resto del flujo.

La idea es empezar por soluciones simples y auditables:

- pesos de clase para la función de pérdida,
- muestreo balanceado por ventanas,
- submuestreo controlado de la clase mayoritaria,
- generación de sesgo artificial para probar robustez,
- métricas de evaluación orientadas a seguridad clínica y fairness.

In [2]:
def compute_class_weights(labels: pd.Series, classes: list[str] = CLASS_ORDER) -> dict[str, float]:
    counts = labels.value_counts().reindex(classes, fill_value=0)
    total = counts.sum()
    weights = {}
    for cls in classes:
        count = counts[cls]
        weights[cls] = float(total / (len(classes) * count)) if count > 0 else 0.0
    return weights

def balanced_window_sample(
    df: pd.DataFrame,
    label_col: str = "class_label",
    per_class: int | None = None,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    parts = []
    for cls in CLASS_ORDER:
        subset = df[df[label_col] == cls]
        if subset.empty:
            continue
        if per_class is None:
            parts.append(subset)
        else:
            n = min(per_class, len(subset))
            parts.append(subset.sample(n=n, random_state=random_state))
    if not parts:
        return df.iloc[0:0].copy()
    out = pd.concat(parts, ignore_index=True)
    return out.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

def stratified_majority_undersample(
    df: pd.DataFrame,
    majority_class: str = "normoglycemia",
    target_ratio: float = 2.0,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    minority_count = int((df["class_label"] != majority_class).sum())
    max_majority = int(target_ratio * max(minority_count, 1))
    maj = df[df["class_label"] == majority_class]
    other = df[df["class_label"] != majority_class]
    maj_sample = maj.sample(n=min(len(maj), max_majority), random_state=random_state) if len(maj) else maj
    return pd.concat([other, maj_sample], ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)

def add_soft_time_augmentations(
    df: pd.DataFrame,
    noise_std: float = 2.0,
    scale_std: float = 0.03,
    shift_max: int = 2,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    out = df.copy()
    if "Measurement" not in out.columns:
        return out
    augmented = out.copy()
    augmented["Measurement"] = augmented["Measurement"] * rng.normal(1.0, scale_std, size=len(augmented))
    augmented["Measurement"] = augmented["Measurement"] + rng.normal(0.0, noise_std, size=len(augmented))
    if "Patient_ID" in augmented.columns and "timestamp" in augmented.columns:
        shift_days = rng.integers(-shift_max, shift_max + 1)
        augmented["timestamp"] = pd.to_datetime(augmented["timestamp"], errors="coerce") + pd.to_timedelta(shift_days, unit="D")
    augmented["class_label"] = augmented["Measurement"].apply(assign_glucose_class)
    return pd.concat([out, augmented], ignore_index=True).sort_values([c for c in ["Patient_ID", "timestamp"] if c in out.columns]).reset_index(drop=True)

def make_balanced_batches(indices: np.ndarray, labels: pd.Series, batch_size: int = 64) -> list[np.ndarray]:
    label_to_indices = {}
    for cls in CLASS_ORDER:
        cls_idx = indices[labels.to_numpy() == cls]
        if len(cls_idx) > 0:
            label_to_indices[cls] = list(cls_idx)
    if not label_to_indices:
        return []
    batches = []
    per_class = max(batch_size // max(len(label_to_indices), 1), 1)
    exhausted = False
    while not exhausted:
        batch = []
        exhausted = True
        for cls, pool in label_to_indices.items():
            if len(pool) == 0:
                continue
            exhausted = False
            take = min(per_class, len(pool))
            batch.extend(pool[:take])
            del pool[:take]
        if batch:
            batches.append(np.array(batch, dtype=int))
    return batches

## 2. Diseño experimental con desbalanceo artificial

Para estudiar si una técnica de balanceo realmente ayuda, conviene introducir desbalance de manera controlada y realista. La clave es no generar un sesgo aleatorio sin interpretación clínica, sino un escenario que pueda ocurrir en práctica.

### Escenarios artificiales sugeridos

1. **Desbalanceo por clase**: reducir hipoglucemias o hiperglucemias en el entrenamiento.
2. **Desbalanceo temporal**: eliminar más eventos nocturnos, postprandiales o de ciertas franjas horarias.
3. **Desbalanceo por paciente**: concentrar muchos eventos en pocos pacientes y dejar el resto con poca cobertura.
4. **Desbalanceo por subgrupo clínico**: infrarepresentar edad, sexo, IMC o centro de procedencia.
5. **Desbalanceo entre datasets**: entrenar con predominio de un dataset y reducir la presencia de los otros para simular shift de dominio.

Estos escenarios permiten medir no solo rendimiento global, sino también robustez, sensibilidad a minorías y comportamiento por subgrupos.

In [3]:
def create_class_imbalance(df: pd.DataFrame, minority_keep: float = 0.3, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    keep_parts = []
    rng = np.random.default_rng(random_state)
    for cls in CLASS_ORDER:
        subset = df[df["class_label"] == cls]
        if cls == "normoglycemia":
            keep_parts.append(subset)
        else:
            n_keep = max(1, int(len(subset) * minority_keep)) if len(subset) else 0
            if n_keep > 0:
                keep_parts.append(subset.sample(n=n_keep, random_state=random_state))
    return pd.concat(keep_parts, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)

def create_temporal_imbalance(
    df: pd.DataFrame,
    night_hours: tuple[int, int] = (0, 6),
    keep_night_ratio: float = 0.4,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    if "timestamp" not in df.columns:
        raise ValueError("Se necesita la columna timestamp para crear desbalance temporal")
    out = df.copy()
    hours = pd.to_datetime(out["timestamp"], errors="coerce").dt.hour
    night_mask = (hours >= night_hours[0]) & (hours < night_hours[1])
    keep_parts = [out[~night_mask]]
    night = out[night_mask]
    if len(night):
        keep_parts.append(night.sample(n=max(1, int(len(night) * keep_night_ratio)), random_state=random_state))
    return pd.concat(keep_parts, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)

def create_patient_imbalance(
    df: pd.DataFrame,
    dominant_patient_share: float = 0.35,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    if "Patient_ID" not in df.columns:
        raise ValueError("Se necesita la columna Patient_ID para crear desbalance por paciente")
    rng = np.random.default_rng(random_state)
    patients = df["Patient_ID"].dropna().unique().tolist()
    if len(patients) < 2:
        return df.copy()
    dominant = rng.choice(patients)
    dominant_df = df[df["Patient_ID"] == dominant]
    rest_df = df[df["Patient_ID"] != dominant]
    rest_keep = rest_df.sample(n=max(1, int(len(rest_df) * (1 - dominant_patient_share))), random_state=random_state) if len(rest_df) else rest_df
    return pd.concat([dominant_df, rest_keep], ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)

def create_subgroup_imbalance(
    df: pd.DataFrame,
    subgroup_col: str,
    minority_keep: float = 0.4,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    if subgroup_col not in df.columns:
        raise ValueError(f"La columna {subgroup_col} no existe en el DataFrame")
    values = df[subgroup_col].dropna().unique().tolist()
    if len(values) < 2:
        return df.copy()
    dominant_value = values[0]
    parts = []
    for value in values:
        subset = df[df[subgroup_col] == value]
        if value == dominant_value:
            parts.append(subset)
        else:
            n_keep = max(1, int(len(subset) * minority_keep)) if len(subset) else 0
            if n_keep > 0:
                parts.append(subset.sample(n=n_keep, random_state=random_state))
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)

def scenario_summary(original_df: pd.DataFrame, transformed_df: pd.DataFrame) -> pd.DataFrame:
    original_counts = original_df["class_label"].value_counts().reindex(CLASS_ORDER, fill_value=0)
    transformed_counts = transformed_df["class_label"].value_counts().reindex(CLASS_ORDER, fill_value=0)
    return pd.DataFrame({
        "class": CLASS_ORDER,
        "original": original_counts.values,
        "transformed": transformed_counts.values,
        "ratio": [
            round(transformed_counts[c] / original_counts[c], 3) if original_counts[c] else np.nan
            for c in CLASS_ORDER
        ]
    })

## 2.1 Framework experimental propuesto

La comparación debe hacerse siempre con el mismo principio:

- el `train` puede modificarse para introducir desbalance artificial y aplicar balanceo,
- `validation` y `test` deben permanecer intactos para medir generalización real,
- el split debe ser por paciente para evitar fuga de información,
- si hay más de un dataset, conviene validar también con un conjunto externo.

### Secuencia recomendada

1. Construir un baseline sin balanceo.
2. Crear uno o varios escenarios de desbalance artificial.
3. Aplicar cada técnica de balanceo solo sobre el entrenamiento.
4. Evaluar con las mismas métricas en todos los casos.
5. Revisar por clase, por paciente y por subgrupo clínico.

In [4]:
def compute_multiclass_metrics(y_true: pd.Series, y_pred: pd.Series, labels: list[str] = CLASS_ORDER) -> dict[str, float]:
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
    }
    return metrics

def compute_minority_metrics(y_true: pd.Series, y_pred: pd.Series, minority_labels: list[str] = ["hypoglycemia", "hyperglycemia"]) -> pd.DataFrame:
    rows = []
    for cls in minority_labels:
        rows.append({
            "class": cls,
            "precision": precision_score(y_true == cls, y_pred == cls, zero_division=0),
            "recall": recall_score(y_true == cls, y_pred == cls, zero_division=0),
            "f1": f1_score(y_true == cls, y_pred == cls, zero_division=0),
        })
    return pd.DataFrame(rows)

def class_event_recall(y_true: pd.Series, y_pred: pd.Series, classes: list[str] = CLASS_ORDER) -> pd.DataFrame:
    rows = []
    for cls in classes:
        mask = y_true == cls
        rows.append({
            "class": cls,
            "event_recall": recall_score(mask, y_pred == cls, zero_division=0),
            "support": int(mask.sum())
        })
    return pd.DataFrame(rows)

def compute_group_fairness(df: pd.DataFrame, y_true_col: str, y_pred_col: str, group_col: str) -> pd.DataFrame:
    if group_col not in df.columns:
        return pd.DataFrame()
    rows = []
    for group_value, group_df in df.groupby(group_col):
        if group_df.empty:
            continue
        y_true = group_df[y_true_col]
        y_pred = group_df[y_pred_col]
        rows.append({
            group_col: group_value,
            "n": len(group_df),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0),
            "minority_recall": recall_score(y_true == "hypoglycemia", y_pred == "hypoglycemia", zero_division=0),
        })
    return pd.DataFrame(rows)

def equal_opportunity_gap(group_metrics: pd.DataFrame, metric_col: str = "minority_recall") -> float:
    if group_metrics.empty or metric_col not in group_metrics.columns:
        return np.nan
    return float(group_metrics[metric_col].max() - group_metrics[metric_col].min())

def summarize_experiment(
    y_true: pd.Series,
    y_pred: pd.Series,
    probas: np.ndarray | None = None,
    group_df: pd.DataFrame | None = None,
    group_col: str | None = None,
) -> dict[str, object]:
    summary = compute_multiclass_metrics(y_true, y_pred)
    minority = compute_minority_metrics(y_true, y_pred)
    event_recall = class_event_recall(y_true, y_pred)
    summary["minority_table"] = minority
    summary["event_recall_table"] = event_recall
    if probas is not None and probas.ndim == 2 and probas.shape[1] >= 2:
        try:
            summary["macro_auprc"] = average_precision_score(pd.get_dummies(y_true, columns=CLASS_ORDER), probas, average="macro")
        except Exception:
            summary["macro_auprc"] = np.nan
    if group_df is not None and group_col is not None and group_col in group_df.columns:
        group_metrics = compute_group_fairness(
            group_df.assign(y_true=y_true.values, y_pred=y_pred.values),
            y_true_col="y_true",
            y_pred_col="y_pred",
            group_col=group_col,
        )
        summary["group_metrics"] = group_metrics
        summary["equal_opportunity_gap"] = equal_opportunity_gap(group_metrics)
    return summary

## 3. Ejemplo de uso: aplicar balanceo y medir el efecto

Esta sección muestra cómo usar las funciones anteriores sobre un conjunto de datos real o, si no está disponible todavía, sobre un ejemplo sintético mínimo. La idea es que puedas ejecutar el flujo completo sin depender de un archivo concreto desde el primer momento.

El objetivo no es entrenar todavía un modelo final, sino verificar que el preprocesamiento y el balanceo están bien definidos y que las métricas permiten comparar escenarios de forma reproducible.

In [5]:
def build_toy_cgm_example(n_patients: int = 8, n_points: int = 240, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    rows = []
    base_time = pd.Timestamp("2024-01-01 00:00:00")
    for pid in range(1, n_patients + 1):
        start = base_time + pd.Timedelta(days=pid)
        glucose = 110 + rng.normal(0, 12, size=n_points).cumsum() * 0.02
        for i in range(n_points):
            ts = start + pd.Timedelta(minutes=5 * i)
            value = float(np.clip(glucose[i] + rng.normal(0, 6), 35, 320))
            rows.append({
                "Patient_ID": f"P{pid:03d}",
                "timestamp": ts,
                "Measurement": value,
                "class_label": assign_glucose_class(value),
                "Sex": "F" if pid % 2 == 0 else "M",
                "AgeGroup": ["child", "young_adult", "adult", "older_adult"][pid % 4],
            })
    toy = pd.DataFrame(rows)
    toy.loc[toy.sample(frac=0.08, random_state=random_state).index, "class_label"] = "hypoglycemia"
    toy.loc[toy.sample(frac=0.10, random_state=random_state + 1).index, "class_label"] = "hyperglycemia"
    return toy

try:
    cgm_df = load_cgm_dataset("T1DiabetesGranada")
    print("Dataset real cargado correctamente.")
except Exception as exc:
    print(f"No se pudo cargar un dataset real todavía: {exc}")
    print("Se usa un ejemplo sintético mínimo para validar las funciones.")
    cgm_df = build_toy_cgm_example()

display(simple_summary(cgm_df).head(12))
print(cgm_df["class_label"].value_counts())

balanced_df = balanced_window_sample(cgm_df, per_class=min(300, cgm_df["class_label"].value_counts().min()), random_state=RANDOM_STATE)
class_weights = compute_class_weights(cgm_df["class_label"])
print("Pesos de clase:", class_weights)
print("Tamaño original:", len(cgm_df), "Tamaño balanceado:", len(balanced_df))

imb_class = create_class_imbalance(cgm_df, minority_keep=0.25)
imb_time = create_temporal_imbalance(cgm_df, keep_night_ratio=0.35)
imb_patient = create_patient_imbalance(cgm_df, dominant_patient_share=0.40)

display(scenario_summary(cgm_df, imb_class))
display(scenario_summary(cgm_df, imb_time))
display(scenario_summary(cgm_df, imb_patient))

augmented_df = add_soft_time_augmentations(balanced_df, noise_std=1.5, scale_std=0.02)
print("Tras augmentations suaves:", len(augmented_df))

Dataset real cargado correctamente.


,column,dtype,nulls,n_unique
0,Patient_ID,str,0,643
1,Measurement_date,object,8244170,1536
2,Measurement_time,object,8244170,1440
3,Measurement,float64,8244170,461
4,15min,datetime64[ns],363919,147390
5,5min,datetime64[ns],30477434,0
6,class_label,str,0,4


class_label
normoglycemia    13274105
missing           8244170
hyperglycemia     7897827
hypoglycemia      1061332
Name: count, dtype: int64
Pesos de clase: {'hypoglycemia': 6.982817817610323, 'normoglycemia': 0.558311690317351, 'hyperglycemia': 0.9383705163458252}
Tamaño original: 30477434 Tamaño balanceado: 900


ValueError: Se necesita la columna timestamp para crear desbalance temporal

## 4. Cómo validar que el balanceo funciona

Para demostrar que una técnica de balanceo es útil en este problema, no basta con mejorar una métrica global. Hay que revisar tres niveles de evidencia:

### 4.1 Mejora en clases minoritarias

- subir `recall` y `F1` en hipoglucemia e hiperglucemia,
- reducir falsos negativos en eventos raros,
- mejorar `PR-AUC` o `average precision` cuando la clase positiva es escasa.

### 4.2 Robustez entre subgrupos

- comparar métricas por edad, sexo, IMC o centro,
- comprobar el peor grupo y no solo la media,
- medir gaps de igualdad de oportunidad entre subgrupos.

### 4.3 Seguridad clínica

- minimizar falsos negativos en eventos de riesgo,
- revisar calibración si el modelo produce probabilidades,
- comprobar que el balanceo no empeora de forma clara los casos de mayor riesgo.

### Regla práctica

Si una técnica mejora el desempeño minoritario pero desploma la precisión global o aumenta mucho las falsas alarmas, no es una buena solución clínica. El objetivo no es equilibrar por estética estadística, sino por utilidad clínica y seguridad.

In [ ]:
def validation_report(
    y_true: pd.Series,
    y_pred: pd.Series,
    group_df: pd.DataFrame | None = None,
    group_col: str | None = None,
) -> dict[str, object]:
    report = {
        "global": compute_multiclass_metrics(y_true, y_pred),
        "minority": compute_minority_metrics(y_true, y_pred),
        "event_recall": class_event_recall(y_true, y_pred),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=CLASS_ORDER),
    }
    if group_df is not None and group_col is not None and group_col in group_df.columns:
        tmp = group_df[[group_col]].copy()
        tmp["y_true"] = y_true.values
        tmp["y_pred"] = y_pred.values
        group_metrics = compute_group_fairness(tmp, y_true_col="y_true", y_pred_col="y_pred", group_col=group_col)
        report["group_metrics"] = group_metrics
        report["equal_opportunity_gap"] = equal_opportunity_gap(group_metrics)
        report["worst_group_balanced_accuracy"] = float(group_metrics["balanced_accuracy"].min()) if not group_metrics.empty else np.nan
    return report

# Plantilla de uso cuando ya tengas predicciones reales
# report = validation_report(y_true=test_df["class_label"], y_pred=test_predictions, group_df=test_df, group_col="Sex")
# display(report["global"])
# display(report["minority"])
# display(report["event_recall"])
# if "group_metrics" in report:
#     display(report["group_metrics"])
#     print("Equal opportunity gap:", report["equal_opportunity_gap"])

## 5. Síntesis operativa

### Orden de trabajo recomendado

1. Cargar y normalizar el dataset.
2. Explorar distribución de clases y subgrupos.
3. Crear uno o varios escenarios de desbalance artificial.
4. Aplicar un baseline sin balanceo.
5. Probar técnicas simples: pesos de clase, balanced batches, submuestreo controlado y augmentations suaves.
6. Evaluar métricas globales, minoritarias y por subgrupo.
7. Elegir la técnica que mejor mejore el rendimiento minoritario sin comprometer la seguridad clínica.

### Decisión práctica

Para este tipo de problema, lo más razonable no es buscar una técnica compleja, sino una combinación simple, trazable y clínicamente justificable. En la mayoría de los casos, la combinación más sólida será:

- pesos de clase o focal loss,
- muestreo balanceado por ventanas,
- control de la distribución por paciente,
- validación por subgrupos,
- y ajuste del umbral de decisión según el coste clínico de los falsos negativos.

Con esto, el notebook queda listo como base metodológica y como punto de partida para implementar experimentos reales sobre T1DiabetesGranada, REPLACE-BG y DiaTrend.